# The Shot Auditor: a trivalent campaign controller on real quantum hardware

**ZEPHIRUM v2.1.0** — sealed-evidence notebook.
Author: **Maria Clara dos Anjos** (SerenaSolutions)

This notebook documents the first demonstration of the **Shot Auditor**: a trivalent controller that decides, after each campaign depth, whether spending further hardware shots is statistically justified — with the decision rule sealed **before** the data.

> **Evidence anchor.** This notebook is the presentation layer of sealed receipt `f910a2b4...` (chained to `0bf0799b...`). The receipt chain, not this document, is the source of truth. Verify with `python -m battery_ci` in the repository root.

**Pre-silicon discipline.** The experimental specification passed an internal adversarial review before any hardware time was spent. Two of its findings were incorporated as the highest-value changes: (1) 1024 shots per energy estimate; (2) a rigorous 95% confidence-interval stopping rule.

## 1. The sealed protocol

The protocol below was sealed (SHA3-256) **before** any data existed. Re-rolling parameters after seeing results would constitute peeking and is forbidden by the protocol itself.

In [ ]:
import json, hashlib, sys
sys.path.insert(0, "src")

receipt = json.load(open("experiments/round_auditor_receipt.json"))
state = json.load(open("experiments/round_auditor_state.json"))
spec = receipt["spec"]
seal = receipt["protocolo"]
print("protocol seal:", seal[:16] + "...")
spec

## 2. The silicon campaign

The ansatz: one parameterized layer applies RX(theta) to 4 qubits followed by a CNOT ring; deeper campaigns repeat the layer. The observable is H = Z0 + Z1Z2, estimated from full-register measurement. The gradient of E with respect to the first-layer angle of qubit 0 is obtained by the parameter-shift rule: dE/dtheta = [E(+pi/2) - E(-pi/2)]/2, each energy estimated with 1024 shots (2048 per depth).

In [ ]:
d1 = state["decisoes"][0]
print("job id:", d1["job"])
print("gradient measured:", d1["g"])
print("95% band:", "+/-", d1["banda"])

## 3. The trivalent decision

The auditor evaluates the sealed rule against the measured data:

In [ ]:
from shot_auditor import audit_step

selo_profundidade = {"spec": {"shots": 2048}}
dec = audit_step(abs(d1["g"]), d1["banda"], 2048, selo_profundidade)
print("decision:", dec["decisao"], "| reason:", dec["motivo"])

## 4. Shot economics

The rule fired at the first depth. Depths 2, 4 and 8 were never executed:

In [ ]:
budget = spec["budget_max"]
spent = 2048
print(f"budget sealed: {budget} shots")
print(f"spent: {spent} shots ({100*spent/budget:.0f}%)")
print(f"saved: {budget - spent} shots ({100*(budget-spent)/budget:.0f}%)")

## 5. Conclusion

At the sealed configuration, the first measured gradient was statistically indistinguishable from zero at the 95% level (|g| = 0.0195 within band ±0.0434). The auditor stopped the campaign with a sealed decision and refused to pay for three further depths.

Two honest caveats, stated up front:

1. **Sequential decision.** Stopping after depth 1 means depths 2-8 were never observed; the sealed rule accepts this, as any sequential test does. The protocol forbids re-rolling the seed (peeking by reroll).
2. **No universality claim.** This is one depth of one ansatz on one 156-qubit superconducting processor with same-day calibration — a demonstration of auditable decision-making, not a general characterization.

Refusals and refutations are published as results, not hidden.

*— Maria Clara dos Anjos, SerenaSolutions / ZEPHIRUM*